# EDA: кандидатогенерация для поиска услуг Авито

Цель — не модель, а ответы на вопросы, от которых зависит архитектура решения (`solution.ipynb`).
Каждый раздел заканчивается выводом с цифрами, полученными на данных.

**Итоги EDA**

| Вопрос | Ответ | Что это значит для решения |
|---|---|---|
| Есть ли объявления train в корпусе бенчмарка? | только 5,3% уникальных объявлений (6,6% строк) | прямая память «запрос → объявление» почти бесполезна; train нужен для статистик и обучения |
| Встречаются ли запросы бенчмарка в train? | 37,5% дословно, 6,7% вместе с локацией | история помогает трети запросов; основную работу делают BM25 и эмбеддинги |
| Насколько важна география? | 83% выборов — в своей локации, медиана 2,9 км, 90% ближе 13 км | поиск ведётся в «гео-множестве» запроса |
| Есть ли у запроса объявления в своей локации? | у 17% запросов бенчмарка — нет ни одного | нужна гео-карта «локация поиска → локации объявлений» (покрывает 100% таких запросов) |
| Предсказуема ли микрокатегория по запросу? | у половины запросов 87,5% выборов — одна микрокатегория | сильный признак из истории |
| Похожа ли валидация на бенчмарк? | запросы бенчмарка длиннее (чаще 3–4 слова против 2), фильтры у 37% против 67% | валидацию подгоняем под бенчмарк по длине и фильтрам |
| Категории | 99% корпуса — категория 114 | фильтровать по категории бесполезно |

Зависимости перечислены в `requirements.txt`. Если они ещё не установлены, раскомментируйте строку ниже.

In [ ]:
# !pip install -r requirements.txt

## 1. Загрузка данных

Файлы ищутся в `AVITO_DATA_DIR` (по умолчанию `./data`), в Colab — в `MyDrive/dataset`. Координаты, цена и рейтинг
приводятся из `Decimal` к `float`; в словарь `SUMMARY` собираются все цифры для итоговой сводки.

In [ ]:
import os, sys, re, json
from decimal import Decimal
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 120)

if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/dataset'
else:
    DATA_DIR = os.environ.get('AVITO_DATA_DIR', './data')

train = pd.read_parquet(f'{DATA_DIR}/train.parquet')
bq    = pd.read_parquet(f'{DATA_DIR}/benchmark_queries.parquet')
bi    = pd.read_parquet(f'{DATA_DIR}/benchmark_items.parquet')

# Координаты, цена и рейтинг в parquet хранятся как Decimal: numpy с ними не работает, приводим к float
def decimals_to_float(df):
    for c in df.columns:
        if df[c].dtype == object:
            s = df[c].dropna().head(100)
            if len(s) and s.map(lambda x: isinstance(x, Decimal)).all():
                df[c] = pd.to_numeric(df[c], errors='coerce').astype('float64')
for d in (train, bq, bi):
    decimals_to_float(d)

SUMMARY = {}
for name, df in [('train', train), ('benchmark_queries', bq), ('benchmark_items', bi)]:
    print(f'{name}: {df.shape}, {df.memory_usage(deep=True).sum()/1e9:.2f} GB')

## 2. Пропуски

Доля пустых значений по колонкам. От пустых описаний и параметров зависит, какие поля брать в текст для поиска.

In [ ]:
def null_share(df):
    # Считаем пустыми и NaN, и пустые строки
    out = {}
    for c in df.columns:
        s = df[c]
        n = s.isna()
        if s.dtype == object:
            n = n | (s.astype(str).str.strip() == '')
        out[c] = round(float(n.mean()), 4)
    return out

nulls = {name: null_share(df) for name, df in [('train', train), ('bq', bq), ('bi', bi)]}
print(pd.DataFrame(nulls).to_string())
SUMMARY['nulls'] = nulls

**Вывод.** В train фильтр пуст у 33% строк, в бенчмарке — у 63%. Рейтинг отсутствует у 6–9% объявлений; описание почти всегда есть.

## 3. Объявления train в корпусе бенчмарка

Если `item_id` есть в train и пересекается с корпусом, можно напрямую брать объявления, которые выбирали по такому же запросу.
Запасной вариант — сопоставить объявления по заголовку, локации и микрокатегории.

In [ ]:
bi_ids = set(bi['item_id'])
if 'item_id' in train.columns:
    tr_ids = set(train['item_id'])
    inter = tr_ids & bi_ids
    SUMMARY['item_id_in_train'] = True
    SUMMARY['train_unique_items'] = len(tr_ids)
    SUMMARY['share_train_items_in_corpus'] = round(len(inter) / len(tr_ids), 4)
    SUMMARY['share_corpus_items_seen_in_train'] = round(len(inter) / len(bi_ids), 4)
    SUMMARY['share_train_rows_item_in_corpus'] = round(float(train['item_id'].isin(bi_ids).mean()), 4)
else:
    SUMMARY['item_id_in_train'] = False

# Запасной вариант: сопоставление по «подписи» объявления (заголовок + локация + микрокатегория)
def sig(df):
    return (df['item_title_raw'].astype(str).str.lower().str.strip() + '|' +
            df['item_location_id'].astype(str) + '|' + df['item_microcat_id'].astype(str))
bi_sig = set(sig(bi))
tr_sig = sig(train)
SUMMARY['share_train_rows_signature_in_corpus'] = round(float(tr_sig.isin(bi_sig).mean()), 4)
SUMMARY['corpus_signature_unique_share'] = round(len(bi_sig) / len(bi), 4)
print({k: v for k, v in SUMMARY.items() if 'item' in k or 'signature' in k})

**Вывод.** `item_id` в train есть, но в корпусе бенчмарка лежат только **5,3%** уникальных объявлений train (**6,6%** строк, ~33 тыс.). Из корпуса в train встречалось 9,6% объявлений. Прямая память «запрос → объявление» почти не работает; эти ~33 тыс. строк становятся основой локальной валидации — поиск по ним идёт в том же корпусе, что и на платформе.

## 4. Запросы бенчмарка в train

Нормализуем текст (нижний регистр, ё → е, только буквы и цифры) и считаем, какая доля запросов бенчмарка встречалась в train:
дословно, вместе с локацией и с точностью до порядка слов.

In [ ]:
def norm_q(s):
    s = str(s).lower().replace('ё', 'е')
    s = re.sub(r'[^0-9a-zа-я]+', ' ', s)
    return ' '.join(s.split())

train['q_norm'] = train['search_query'].map(norm_q)
bq['q_norm']    = bq['search_query'].map(norm_q)

train_q = set(train['q_norm'])
train_q_loc = set(zip(train['q_norm'], train['search_location_id']))

SUMMARY['bq_query_seen_in_train'] = round(float(bq['q_norm'].isin(train_q).mean()), 4)
SUMMARY['bq_query_loc_seen_in_train'] = round(float(pd.Series(list(zip(bq['q_norm'], bq['search_location_id']))).isin(train_q_loc).mean()), 4)

# Совпадение по отсортированному набору слов (ловит перестановки: "ремонт стиральных машин" / "стиральных машин ремонт")
bag = lambda q: ' '.join(sorted(q.split()))
train_bag = set(train['q_norm'].map(bag))
SUMMARY['bq_query_bag_seen_in_train'] = round(float(bq['q_norm'].map(bag).isin(train_bag).mean()), 4)

SUMMARY['train_unique_queries'] = int(train['q_norm'].nunique())
SUMMARY['bq_unique_queries'] = int(bq['q_norm'].nunique())
print({k: v for k, v in SUMMARY.items() if k.startswith(('bq_', 'train_unique_q'))})

**Вывод.** 2451 уникальный текст на 2452 запроса бенчмарка — «длинный хвост». Дословно в train встречается **37,5%** запросов, вместе с локацией — **6,7%**. В train медианный запрос встречается 1 раз, топ-1000 запросов дают 53% строк.

## 5. Сколько объявлений выбирают на один поиск

В train нет `query_id`, поэтому «поиском» считаем группу строк с одинаковыми признаками запроса. Это приближение,
но оно показывает, одно или несколько релевантных объявлений обычно бывает.

In [ ]:
q_cols = [c for c in ['search_query', 'search_location_id', 'search_is_delivery_search',
                      'search_infm_params_text', 'search_category'] if c in train.columns]
grp = train.groupby(q_cols, dropna=False).size()
SUMMARY['positives_per_search_group'] = grp.describe(percentiles=[.5, .9, .99]).round(2).to_dict()
SUMMARY['train_duplicate_rows_share'] = round(float(train.duplicated().mean()), 4)
print(SUMMARY['positives_per_search_group'])

**Вывод.** В среднем 1,4 выбранных объявления на поиск, медиана — 1, 99-й перцентиль — 7.

## 6. География

Доля выбранных объявлений в той же локации, что и поиск. Координат поиска нет, поэтому центр локации оцениваем как медиану
координат объявлений корпуса в ней и считаем расстояние до выбранного объявления.

In [ ]:
SUMMARY['train_same_location_share'] = round(float((train['item_location_id'] == train['search_location_id']).mean()), 4)

# Центры локаций по корпусу
centers = bi.groupby('item_location_id')[['item_latitude', 'item_longitude']].median()

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

t = train.join(centers, on='search_location_id', rsuffix='_c')
m = t['item_latitude_c'].notna()
d = haversine(t.loc[m, 'item_latitude'], t.loc[m, 'item_longitude'],
              t.loc[m, 'item_latitude_c'], t.loc[m, 'item_longitude_c'])
SUMMARY['train_search_loc_has_center_share'] = round(float(m.mean()), 4)
SUMMARY['train_dist_km_quantiles'] = pd.Series(d).quantile([.5, .75, .9, .95, .99]).round(1).to_dict()
SUMMARY['delivery_search_share_train'] = round(float(train['search_is_delivery_search'].mean()), 4)

# Для удалённых услуг (доставка) гео должно быть слабее: сравним
for flag in [0, 1]:
    s = train[train['search_is_delivery_search'] == flag]
    SUMMARY[f'same_loc_share_delivery_{flag}'] = round(float((s['item_location_id'] == s['search_location_id']).mean()), 4) if len(s) else None
print({k: v for k, v in SUMMARY.items() if 'loc' in k or 'dist' in k or 'deliv' in k})

**Вывод.** **83%** выборов — объявление из той же локации, что и поиск; медианное расстояние до выбранного объявления **2,9 км**, 90% — ближе 13 км. Флаг доставки в train всегда 0. У медианного запроса бенчмарка в своей локации ~1,6 тыс. объявлений (вместо 189 тыс.), но у **17%** — ни одного (обычно поиск по региону, а объявления привязаны к городам).

## 7. Категории и микрокатегории

Если по запросу почти всегда выбирают одну микрокатегорию, её можно использовать как сильный фильтр или признак.

In [ ]:
# Доля выборов, приходящаяся на самую частую микрокатегорию запроса (только запросы с >=5 строками)
g = train.groupby(['q_norm', 'item_microcat_id']).size().rename('n').reset_index()
tot = g.groupby('q_norm')['n'].transform('sum')
g['share'] = g['n'] / tot
top = g[tot >= 5].groupby('q_norm')['share'].max()
SUMMARY['query_top_microcat_share_quantiles'] = top.quantile([.1, .25, .5, .75]).round(3).to_dict()
SUMMARY['n_microcats'] = {'train': int(train['item_microcat_id'].nunique()), 'bi': int(bi['item_microcat_id'].nunique())}
SUMMARY['n_categories'] = {'train': int(train['item_category_id'].nunique()), 'bi': int(bi['item_category_id'].nunique())}

# Как search_category соотносится с категорией объявления
print(pd.crosstab(train['search_category'], train['item_category_id']).iloc[:15, :15])
print(SUMMARY['query_top_microcat_share_quantiles'])

**Вывод.** У половины запросов **87,5%** выборов приходится на одну микрокатегорию — это сильный сигнал. В train 212 микрокатегорий, в корпусе 752.

## 8. Фильтры поиска и параметры объявлений

Как часто есть фильтр и какие фильтры самые частые. Если текст фильтра дословно встречается в параметрах объявления,
это готовый признак совпадения.

In [ ]:
has_f = lambda s: s.fillna('').astype(str).str.strip() != ''
SUMMARY['filter_share'] = {'train': round(float(has_f(train['search_infm_params_text']).mean()), 4),
                           'bq': round(float(has_f(bq['search_infm_params_text']).mean()), 4)}
print(train.loc[has_f(train['search_infm_params_text']), 'search_infm_params_text'].value_counts().head(15).to_string())
print()
print(bi['item_infm_params_text'].dropna().sample(5, random_state=0).to_string())

# Совпадение фильтра с параметрами объявления (грубая проверка по вхождению подстроки)
sub = train[has_f(train['search_infm_params_text'])].sample(min(20000, int(has_f(train['search_infm_params_text']).sum())), random_state=0)
hit = [str(f).lower() in str(p).lower() for f, p in zip(sub['search_infm_params_text'], sub['item_infm_params_text'])]
SUMMARY['filter_substring_in_item_params_share'] = round(float(np.mean(hit)), 4) if len(hit) else None
print(SUMMARY['filter_share'], SUMMARY['filter_substring_in_item_params_share'])

**Вывод.** Фильтры — это «Вид услуги …» и «Тип услуги …» из закрытых списков Авито; дословно в параметрах выбранного объявления фильтр встречается в 54% случаев.

## 9. Похожи ли запросы бенчмарка на train

Сравниваем длину запроса в словах. Локальная валидация должна повторять эти распределения.

In [ ]:
qlen = lambda s: s.map(lambda x: len(x.split()))
SUMMARY['query_len_words'] = {
    'train': qlen(train['q_norm']).value_counts(normalize=True).sort_index().head(8).round(3).to_dict(),
    'bq':    qlen(bq['q_norm']).value_counts(normalize=True).sort_index().head(8).round(3).to_dict(),
}
SUMMARY['bq_search_location_seen_in_train'] = round(float(bq['search_location_id'].isin(set(train['search_location_id'])).mean()), 4)
if 'search_category' in bq.columns:
    SUMMARY['search_category_dist'] = {
        'train': train['search_category'].value_counts(normalize=True).head(8).round(3).to_dict(),
        'bq':    bq['search_category'].value_counts(normalize=True).head(8).round(3).to_dict(),
    }
print(SUMMARY['query_len_words'])

**Вывод.** Запросы бенчмарка длиннее: в train чаще всего 2 слова (45%), в бенчмарке — 3 (32%) и 4 (21%). Все локации бенчмарка встречаются в train.

## 10. Длина текстов объявлений

От длины зависят `max_seq_length` эмбеддера и то, сколько описания брать в BM25.

In [ ]:
for c in ['item_title_raw', 'item_description_raw', 'item_infm_params_text']:
    SUMMARY[f'len_chars_{c}'] = bi[c].fillna('').astype(str).str.len().quantile([.5, .9, .99]).round(0).to_dict()
print({k: v for k, v in SUMMARY.items() if k.startswith('len_chars')})

**Вывод.** Заголовок — медиана 35 символов, описание — 1054, параметры — 738 (в параметрах лежат вид и тип услуги, адрес, цены). Для эмбеддингов берём заголовок + начало параметров и описания.

## 11. Примеры запросов бенчмарка

Несколько случайных запросов, чтобы увидеть их «на глаз», и отдельно запросы, которых нет в train.

In [ ]:
SUMMARY['bq_examples'] = bq['search_query'].sample(15, random_state=1).tolist()
SUMMARY['bq_unseen_examples'] = bq.loc[~bq['q_norm'].isin(train_q), 'search_query'].sample(
    min(15, int((~bq['q_norm'].isin(train_q)).sum())), random_state=1).tolist()
print(SUMMARY['bq_examples']); print(SUMMARY['bq_unseen_examples'])

**Вывод.** Примеры запросов: «надувные шары с доставкой на 1 сентября», «ремонт капсульных кофемашин nespresso», «репетитор начальных классов метро алтуфьево» — длинные, конкретные, часто с опечатками («ремонт инвернторных микроволновых печей»).

## 12. Дополнительные проверки

Какая часть корпуса относится к категориям train, как обрабатывать запросы без объявлений в своей локации
и сколько в train строк с объявлениями из корпуса.

In [ ]:
EXTRA = {}
bi_ids = set(bi['item_id'])
EXTRA['corpus_share_in_train_categories'] = round(float(bi['item_category_id'].isin(set(train['item_category_id'])).mean()), 4)
EXTRA['corpus_top_categories'] = bi['item_category_id'].value_counts(normalize=True).head(5).round(3).to_dict()

# Запросы без объявлений в своей локации: покрывает ли их карта «локация поиска -> локации объявлений» из train
loc_map = train.groupby('search_location_id')['item_location_id'].agg(lambda s: s.value_counts().head(5).index.tolist())
no_local = bq[~bq['search_location_id'].isin(set(bi['item_location_id']))]
EXTRA['bq_no_local_items_n'] = int(len(no_local))
EXTRA['bq_no_local_covered_by_loc_map'] = round(float(no_local['search_location_id'].isin(loc_map.index).mean()), 4)
top5 = train['search_location_id'].map(loc_map)
EXTRA['train_item_loc_in_top5_map'] = round(float(np.mean([il in t for il, t in zip(train['item_location_id'], top5)])), 4)

in_c = train['item_id'].isin(bi_ids)
EXTRA['train_rows_item_in_corpus'] = int(in_c.sum())
EXTRA['train_in_corpus_unique_queries'] = int(train.loc[in_c, 'q_norm'].nunique())
print(json.dumps(EXTRA, ensure_ascii=False, indent=1))

**Вывод.** 99% корпуса — категория 114, фильтровать по категории бесполезно. Все 427 запросов бенчмарка без объявлений в своей
локации покрываются гео-картой из train, а в топ-5 локаций по карте попадают **95,6%** выборов. Строк train с объявлением из корпуса —
33 010 (12 169 уникальных запросов): из них собирается локальная валидация и обучающая выборка для ранжировщика.

## 13. Итоговая сводка

Все цифры, собранные выше в `SUMMARY`, одним JSON.

In [ ]:
def to_jsonable(o):
    if isinstance(o, dict):  return {str(k): to_jsonable(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [to_jsonable(v) for v in o]
    if isinstance(o, (np.integer,)):  return int(o)
    if isinstance(o, (np.floating,)): return float(o)
    return o

print(json.dumps(to_jsonable(SUMMARY), ensure_ascii=False, indent=1))